# Crop Viewer

For a given PDF, exports every image crop stored in the index to a local folder — organised by page, labelled by crop type.

Reads directly from the ChromaDB vector store and the kvstore (HDF5). No service needs to be running.

In [ ]:
import os
from pathlib import Path
from collections import defaultdict, Counter

import chromadb
from chromadb.config import Settings

from colette.kvstore import ImageStorageFactory

## Configuration

In [ ]:
colette_root = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))

# ── User configuration ───────────────────────────────────────────────────────
app_dir       = colette_root / 'YOUR_APP_DIR'    # ← directory passed to service_create
documents_dir = colette_root / 'YOUR_DOCS_DIR'   # ← folder containing the indexed PDFs

# Target PDF — basename only (directory is resolved automatically).
# Run the "List all indexed PDFs" cell below to discover available filenames.
target_pdf = 'your_document.pdf'               # ← set to the PDF you want to inspect
# ─────────────────────────────────────────────────────────────────────────────

# Output folder — crops are saved to: <export_root>/<pdf_stem>/page_NNN/<label>_NNN.png
export_root = colette_root / 'tmp' / 'crops'

source_path = str(documents_dir / target_pdf)
export_dir  = export_root / Path(target_pdf).stem

print(f'App dir    : {app_dir}')
print(f'Source path: {source_path}')
print(f'PDF exists : {Path(source_path).exists()}')
print(f'Export to  : {export_dir}')

## Open the index

In [ ]:
index_path   = app_dir / 'mm_index'
kvstore_path = app_dir / 'kvstore.db'

chroma_client = chromadb.Client(
    Settings(
        is_persistent=True,
        persist_directory=str(index_path),
        anonymized_telemetry=False,
    )
)
collection = chroma_client.get_collection(name='mm_db')
kvstore    = ImageStorageFactory.create_storage('hdf5', kvstore_path, mode='r')

print(f'Total crops in index : {collection.count()}')

## Helper — list all indexed PDFs

Run this cell to discover what PDFs are in the index and find the exact filename you need.

In [ ]:
sample  = collection.get(limit=10000, include=['metadatas'])
sources = sorted({Path(m['source']).name for m in sample['metadatas'] if m.get('source')})
print(f'{len(sources)} unique PDFs in index:')
for s in sources:
    print(' ', s)

## Query crops for the target PDF

In [ ]:
results = collection.get(
    where={'source': source_path},
    include=['metadatas'],
)

ids       = results['ids']
metadatas = results['metadatas']

print(f'Crops found for "{target_pdf}": {len(ids)}')
if not ids:
    print('\nNo crops found. Check that:')
    print('  1. The PDF was indexed (run service_index first).')
    print('  2. The filename matches exactly (run the "List all indexed PDFs" cell).')
    print(f'  3. The source path matches what is stored: {source_path}')

## Export crops to disk

Saves each crop as a PNG file under `<export_root>/<pdf_stem>/page_NNN/<label>_NNN.png`.

In [ ]:
if not ids:
    print('No crops to export.')
else:
    # Group by page
    pages = defaultdict(list)
    for crop_id, meta in zip(ids, metadatas):
        page = meta.get('page_number') or 0
        pages[page].append((crop_id, meta))

    exported = 0
    missing  = 0

    for page_num in sorted(pages):
        page_dir = export_dir / f'page_{page_num:03d}'
        page_dir.mkdir(parents=True, exist_ok=True)

        crops = pages[page_num]
        label_counter = Counter()

        for crop_id, meta in crops:
            label = meta.get('crop_label') or 'unknown'
            idx   = label_counter[label]
            label_counter[label] += 1
            out_path = page_dir / f'{label}_{idx:03d}.png'

            try:
                image = kvstore.retrieve_image(crop_id)
                image.load()          # force full decode before buffer can be GC'd
                image.save(out_path, format='PNG')
                exported += 1
            except KeyError:
                print(f'  [missing in kvstore] page={page_num} id={crop_id}')
                missing += 1

    print(f'Exported : {exported} crops')
    if missing:
        print(f'Missing  : {missing} crops (key not found in kvstore)')
    print(f'Location : {export_dir}')

## Summary

In [ ]:
if ids:
    label_counts = Counter(m.get('crop_label') or 'unknown' for m in metadatas)
    print(f'PDF      : {target_pdf}')
    print(f'Pages    : {len(pages)}')
    print(f'Crops    : {len(ids)}')
    print(f'By type  :')
    for label, count in label_counts.most_common():
        print(f'  {label:<20} {count}')
    print()
    print('Page breakdown:')
    for page_num in sorted(pages):
        page_labels = Counter(m.get('crop_label') or 'unknown' for _, m in pages[page_num])
        detail = ', '.join(f'{l}:{n}' for l, n in page_labels.most_common())
        print(f'  page {page_num:>3}  {len(pages[page_num]):>3} crop(s)  [{detail}]')

In [ ]:
kvstore.close()